In [2]:
import torch
import cv2
import glob
import numpy as np

from repositories.LPRNet_Pytorch.model.LPRNet import LPRNet

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = LPRNet(
    lpr_max_len=8,
    phase=False,
    class_num=37,   # 0-9 + A-Z + blank
    dropout_rate=0.5
).to(device)

model.eval()

images = glob.glob("runs/crops/*.jpg")

print("TOTAL CROPS:", len(images))

for img_path in images:
    img = cv2.imread(img_path)
    img = cv2.resize(img, (94, 24))
    img = img / 255.0
    img = np.transpose(img, (2, 0, 1))
    img = torch.tensor(img, dtype=torch.float32).unsqueeze(0).to(device)

    with torch.no_grad():
        preds = model(img)

    print(img_path, preds)

TOTAL CROPS: 70
runs/crops\crop_0_0.jpg tensor([[[-4.4568e+00, -4.4695e+00, -4.4204e+00, -4.4188e+00, -4.3773e+00,
          -4.4385e+00, -4.5070e+00, -4.4200e+00, -4.5338e+00, -4.4553e+00,
          -4.5002e+00, -4.4408e+00, -4.5172e+00, -4.4841e+00, -4.5201e+00,
          -4.5996e+00, -4.6043e+00, -4.5664e+00],
         [-4.5661e-01, -5.3795e-01, -5.0891e-01, -5.2119e-01, -4.4323e-01,
          -5.3930e-01, -5.7661e-01, -5.1628e-01, -6.2914e-01, -5.5212e-01,
          -5.8802e-01, -5.1169e-01, -5.8336e-01, -5.0375e-01, -5.6581e-01,
          -6.1058e-01, -6.4924e-01, -6.8622e-01],
         [-1.2638e+01, -1.2701e+01, -1.2679e+01, -1.2659e+01, -1.2679e+01,
          -1.2683e+01, -1.2661e+01, -1.2683e+01, -1.2655e+01, -1.2688e+01,
          -1.2653e+01, -1.2690e+01, -1.2671e+01, -1.2717e+01, -1.2725e+01,
          -1.2689e+01, -1.2705e+01, -1.2635e+01],
         [ 4.7467e-01,  5.1298e-01,  5.0681e-01,  4.7641e-01,  5.1061e-01,
           5.7592e-01,  4.4643e-01,  5.4241e-01,  4.8345e-01

In [3]:
import cv2
import glob
import torch
import numpy as np

from repositories.LPRNet_Pytorch.model.LPRNet import LPRNet

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# model
model = LPRNet(
    lpr_max_len=8,
    phase=False,
    class_num=37,
    dropout_rate=0.5
).to(device)

model.load_state_dict(torch.load(
    r"C:\Users\lindo\Downloads\ALPR-RU\repositories\LPRNet_Pytorch\weights\Final_LPRNet_model.pth",
    map_location=device
))

model.eval()

CHARS = "0123456789ABCDEFGHIJKLMNOPQRSTUVWXYZ"

def decode(preds):
    preds = preds.cpu().detach().numpy()
    out = []

    for p in preds:
        p = np.argmax(p, axis=1)

        last = -1
        s = ""
        for i in p:
            if i != last and i != 0:
                s += CHARS[i-1]
            last = i

        out.append(s)

    return out


images = glob.glob("runs/crops/*.jpg")

for img_path in images:
    img = cv2.imread(img_path)
    img = cv2.resize(img, (94, 24))
    img = img.astype(np.float32) / 255.0
    img = np.transpose(img, (2, 0, 1))
    img = torch.tensor(img).unsqueeze(0).to(device)

    with torch.no_grad():
        preds = model(img)

    text = decode(preds)

    print(img_path, "→", text)

C:\Users\lindo\AppData\Local\Temp\ipykernel_11292\1593434530.py:18: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  model.load_state_dict(torch.load(


RuntimeError: Error(s) in loading state_dict for LPRNet:
	size mismatch for backbone.20.weight: copying a param with shape torch.Size([68, 256, 13, 1]) from checkpoint, the shape in current model is torch.Size([37, 256, 13, 1]).
	size mismatch for backbone.20.bias: copying a param with shape torch.Size([68]) from checkpoint, the shape in current model is torch.Size([37]).
	size mismatch for backbone.21.weight: copying a param with shape torch.Size([68]) from checkpoint, the shape in current model is torch.Size([37]).
	size mismatch for backbone.21.bias: copying a param with shape torch.Size([68]) from checkpoint, the shape in current model is torch.Size([37]).
	size mismatch for backbone.21.running_mean: copying a param with shape torch.Size([68]) from checkpoint, the shape in current model is torch.Size([37]).
	size mismatch for backbone.21.running_var: copying a param with shape torch.Size([68]) from checkpoint, the shape in current model is torch.Size([37]).
	size mismatch for conta